# NLP Masterclass 01: Text Preprocessing & Tokenization

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** Python fundamentals, Basic Probability  
**Difficulty:** ⭐ Beginner → Intermediate  
**Estimated Time:** 90–120 minutes

---



## 🎓 Socratic Deep Check

> *"GPT-4 has a vocabulary of ~100,256 tokens. Why is this number so specific? If we used raw characters, our sequences would be 5x longer. If we used whole words, our vocabulary would be 1,000,000+ words. How do we find the 'Goldilocks' zone between computational efficiency and linguistic expressive power?"*

---

## Why This Matters for AI Engineers

Every LLM interaction starts with tokenization. **API costs are per-token.** Context windows are measured in tokens. RAG chunking (NB21) depends on token boundaries. Even 'traditional' AI tasks like sentiment analysis or classification often fail not because the model is weak, but because the preprocessing was noisy or inconsistent.


---

## 📑 Table of Contents

* [🎓 Socratic Deep Check](#-socratic-deep-check)
* [0 · Setup & Prerequisites](#0-setup-prerequisites)
* [1 · Traditional Preprocessing (The Pre-LLM Era)](#1-traditional-preprocessing-the-pre-llm-era)
  * [1.1 · The Production-Grade Cleaning Pipeline](#11-the-production-grade-cleaning-pipeline)
  * [1.2 · Stemming vs. Lemmatization](#12-stemming-vs-lemmatization)
* [2 · NLP Data Augmentation (EDA)](#2-nlp-data-augmentation-eda)
  * [2.1 · Why Text Augmentation is Harder](#21-why-text-augmentation-is-harder)
  * [2.2 · Easy Data Augmentation (EDA) Techniques](#22-easy-data-augmentation-eda-techniques)
  * [2.3 · Advanced: Back-Translation](#23-advanced-back-translation)
* [3 · Classical Statistical NLP (The Vector Space Era)](#3-classical-statistical-nlp-the-vector-space-era)
  * [3.1 · Bag-of-Words (BoW)](#31-bag-of-words-bow)
  * [3.2 · TF-IDF: The Math](#32-tf-idf-the-math)
* [4 · Modern Subword Tokenization](#4-modern-subword-tokenization)
  * [4.1 · Byte-Pair Encoding (BPE)](#41-byte-pair-encoding-bpe)
  * [4.2 · WordPiece vs. Unigram (SentencePiece)](#42-wordpiece-vs-unigram-sentencepiece)
  * [4.3 · Subword Regularization for Robustness](#43-subword-regularization-for-robustness)
* [5 · Production Hands-on: Building a Tokenizer](#5-production-hands-on-building-a-tokenizer)
* [✅ Knowledge Check](#-knowledge-check)
* [🔨 Practical Challenge](#-practical-challenge)

---


## 0 · Setup & Prerequisites

In [ ]:
# 1. Install necessary libraries
!pip install -q nltk spacy scikit-learn transformers tokenizers tiktoken

# 2. Download NLTK data and spaCy model
import nltk
import spacy
import subprocess

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

try:
    nlp = spacy.load('en_core_web_sm')
except OSError:
    subprocess.run(['python', '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

print("✅ Setup Complete: Dependencies installed and models downloaded.")

---
## 1 · Traditional Preprocessing (The Pre-LLM Era)

Before we look at learned tokenization, we must handle the raw noise in human text. While modern LLMs are robust, traditional ML and search engines (ElasticSearch/OpenSearch) still rely heavily on these deterministic filters.

### 1.1 · The Production-Grade Cleaning Pipeline
We use **Unicode Normalization** to handle different encodings of the same character (e.g., 'e' + '´' vs 'é').

In [ ]:
import re
import unicodedata
from typing import Optional

def production_clean(text: str, aggressive: bool = False) -> str:
    """
    A robust cleaning function for production pipelines.
    
    Args:
        text (str): The raw input string.
        aggressive (bool): If True, removes all non-alphanumeric characters. Defaults to False.
        
    Returns:
        str: The normalized and cleaned text.
    """
    # 1. Unicode Normalization (NFKD separates base chars from accents)
    text = unicodedata.normalize('NFKD', text)
    
    # 2. Lowercasing (Standardize casing)
    text = text.lower()
    
    # 3. Strip HTML Tags (Regex-based)
    text = re.sub(r'<[^>]*>', ' ', text)
    
    # 4. Handle Emojis & Weird Punctuation (Optional)
    if aggressive:
        # Remove everything except alphanumeric and spaces
        text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    
    # 5. Collapse Whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# TEST IT
raw_input = "  <p>The résumé was <b>PERFECT</b>! 🎉  \n Contact: me@example.com </p> "
print(f"Raw: {raw_input}")
print(f"Cleaned (Balanced):  {production_clean(raw_input)}")
print(f"Cleaned (Aggressive): {production_clean(raw_input, aggressive=True)}")

### 1.2 · Stemming vs. Lemmatization

| Feature | Stemming (e.g., Porter) | Lemmatization (e.g., spaCy) |
|---------|-------------------------|---------------------------|
| **Logic** | Hops off suffixes (Heuristic) | Uses dictionary & context (Morphological) |
| **Speed** | 🚀 Extremely Fast | 🐢 Slower (requires POS tagging) |
| **Accuracy**| Low ("studies" → "studi") | High ("studies" → "study") |
| **Usecase** | High-speed indexing, search | Complex NLP, Chatbots |

In [ ]:
from nltk.stem import PorterStemmer
import spacy
from typing import List

def compare_reduction(words: List[str]) -> None:
    """
    Compares Stemming and Lemmatization for a list of words.
    """
    stemmer = PorterStemmer()
    nlp = spacy.load("en_core_web_sm")

    print(f"{ 'Word':<12} | { 'Stemming':<12} | { 'Lemmatization':<12}")
    print("-" * 45)
    for w in words:
        stem = stemmer.stem(w)
        lemma = nlp(w)[0].lemma_
        print(f"{w:<12} | {stem:<12} | {lemma:<12}")

test_words = ["changing", "changes", "changed", "charger", "charge"]
compare_reduction(test_words)

---
## 2 · NLP Data Augmentation (EDA)

In Computer Vision, augmentation is straightforward: rotate a cat, and it's still a cat. In NLP, flipping a word or changing a character can shift the entire sentiment or meaning. 

### 🎓 Socratic Deep Check
> *"If I replace 'good' with 'bad' in a movie review, the grammar is perfect, but the label is now wrong. If I replace 'good' with 'great', the label is preserved. How do we automate this without a human in the loop?"*

### 2.1 · Why Text Augmentation is Harder
1. **Discreteness:** Images are continuous pixels; text is discrete tokens. There is no "middle ground" between two words.
2. **Semantic Sensitivity:** Small changes (e.g., adding "not") = big meaning shifts.
3. **Syntactic Fragility:** Swapping words blindly can make a sentence ungrammatical.

### 2.2 · Easy Data Augmentation (EDA) Techniques
Wei and Zou (2019) proposed four simple operations to boost performance on small datasets: **Synonym Replacement**, **Random Insertion**, **Random Swapping**, and **Random Deletion**.

In [ ]:
import random
from typing import List, Set
from nltk.corpus import wordnet

class TextAugmentor:
    """
    A production-grade implementation of Easy Data Augmentation (EDA) techniques.
    """
    
    def get_synonyms(self, word: str) -> List[str]:
        """
        Find synonyms using WordNet.
        
        Args:
            word (str): The word to find synonyms for.
        Returns:
            List[str]: A list of unique synonyms.
        """
        synonyms = set()
        for syn in wordnet.synsets(word):
            for l in syn.lemmas():
                synonym = l.name().replace('_', ' ').replace('-', ' ').lower()
                if synonym != word.lower():
                    synonyms.add(synonym)
        return list(synonyms)

    def synonym_replacement(self, sentence: str, n: int = 1) -> str:
        """
        Replaces n random non-stop words with their synonyms.
        """
        words = sentence.split()
        new_words = words.copy()
        random_word_list = list(set([word for word in words if word.isalnum()]))
        random.shuffle(random_word_list)
        
        num_replaced = 0
        for random_word in random_word_list:
            synonyms = self.get_synonyms(random_word)
            if len(synonyms) >= 1:
                synonym = random.choice(synonyms)
                new_words = [synonym if word == random_word else word for word in new_words]
                num_replaced += 1
            if num_replaced >= n:
                break

        return ' '.join(new_words)

    def random_deletion(self, sentence: str, p: float = 0.2) -> str:
        """
        Randomly deletes words with probability p.
        """
        words = sentence.split()
        if len(words) == 1: return sentence
        
        new_words = [w for w in words if random.uniform(0, 1) > p]
        
        # Ensure at least one word remains
        if len(new_words) == 0:
            return random.choice(words)
        return ' '.join(new_words)

augmentor = TextAugmentor()
sample = "The product was incredibly easy to use and fast."

print(f"Original: {sample}")
print(f"Synonym Replacement: {augmentor.synonym_replacement(sample, n=2)}")
print(f"Random Deletion:      {augmentor.random_deletion(sample, p=0.3)}")

### 2.3 · Advanced: Back-Translation

The gold standard for semantic-preserving augmentation. We translate a sentence to a pivot language (e.g., German) and then back to the original language. This often results in a sentence with the same meaning but different phrasing.

> **Why Back-Translation?** It captures syntactic variety and paraphrasing that simple word swapping cannot.

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
from typing import Tuple
import torch

def get_marian_bt_models(src_lang: str = "en", pivot_lang: str = "de") -> Tuple[MarianTokenizer, MarianMTModel, MarianTokenizer, MarianMTModel]:
    """
    Utility to load source -> pivot and pivot -> source translation models.
    """
    fwd_model_name = f"Helsinki-NLP/opus-mt-{src_lang}-{pivot_lang}"
    rev_model_name = f"Helsinki-NLP/opus-mt-{pivot_lang}-{src_lang}"
    
    fwd_tokenizer = MarianTokenizer.from_pretrained(fwd_model_name)
    fwd_model = MarianMTModel.from_pretrained(fwd_model_name)
    
    rev_tokenizer = MarianTokenizer.from_pretrained(rev_model_name)
    rev_model = MarianMTModel.from_pretrained(rev_model_name)
    
    return fwd_tokenizer, fwd_model, rev_tokenizer, rev_model

def back_translate(text: str, fwd_tok, fwd_mod, rev_tok, rev_mod) -> str:
    """
    Performs EN -> DE -> EN back-translation.
    """
    # 1. English to German
    inputs = fwd_tok(text, return_tensors="pt", padding=True)
    with torch.no_grad():
        translated = fwd_mod.generate(**inputs)
    german_text = fwd_tok.decode(translated[0], skip_special_tokens=True)
    
    # 2. German back to English
    inputs = rev_tok(german_text, return_tensors="pt", padding=True)
    with torch.no_grad():
        back_translated = rev_mod.generate(**inputs)
    return rev_tok.decode(back_translated[0], skip_special_tokens=True)

print("⏳ Loading Back-Translation models (Helsinki-NLP)...")
# Note: These models represent ~600MB of downloads.
# fwd_t, fwd_m, rev_t, rev_m = get_marian_bt_models()
# bt_sample = "The quick brown fox jumps over the lazy dog."
# print(f"Original: {bt_sample}")
# print(f"Back-Translated: {back_translate(bt_sample, fwd_t, fwd_m, rev_t, rev_m)}")

---
## 3 · Classical Statistical NLP (The Vector Space Era)

How do we turn words into numbers without deep learning? For decades, **TF-IDF** was the gold standard for information retrieval.

### 3.1 · Bag-of-Words (BoW)
A simple count of word frequencies. 
**Problem:** It treats "the" (very common, low info) as more important than "quantum" (rare, high info).

### 3.2 · TF-IDF: The Math

$$TF(t, d) = \frac{\text{Count of term } t \text{ in doc } d}{\text{Total words in doc } d}$$

$$IDF(t, D) = \log \left( \frac{\text{Total docs in } D}{1 + \text{Docs containing } t} \right)$$

$$TF\text{-}IDF = TF \times IDF$$

Why the log? It dampens the effect of frequency, ensuring that having a term 100 times isn't 100x more important than having it once.

In [ ]:
import math
from collections import Counter
import pandas as pd
from typing import Dict, List
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = [
    "the cat sat on the mat",
    "the dog sat on the rug",
    "the cat played with the dog"
]

def calculate_tfidf_scratch(documents: List[str]) -> Dict[str, float]:
    """
    Calculates TF-IDF for the first document in a corpus for demonstration.
    """
    # 1. Collect all terms
    all_words = set(" ".join(documents).split())
    N = len(documents)
    
    # 2. IDF calculation
    idf = {}
    for word in all_words:
        doc_count = sum(1 for d in documents if word in d.split())
        idf[word] = math.log(N / (doc_count))
    
    # 3. TF-IDF Calculation for the first document
    doc0 = documents[0].split()
    counts = Counter(doc0)
    results = {}
    for word in set(doc0):
        tf = counts[word] / len(doc0)
        results[word] = tf * idf[word]
    return results

print("TF-IDF (Scratch) for Doc 0:")
print(calculate_tfidf_scratch(corpus))

# The Scikit-Learn Way
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
print("\nSklearn Vocabulary Size:", len(vectorizer.get_feature_names_out()))

### 🎓 Concept Bridge: The Sparsity Problem

**The Senior Perspective:** Imagine a vocabulary of 50,000 words. Every sentence in your dataset is now represented as a vector of length 50,000. 
*   99.9% of that vector will be zeros. 
*   This is **The Curse of Dimensionality**. 
*   Models struggle to learn patterns when the input space is so sparse.

**This is precisely why we transitioned to Embeddings (Dense Vectors) and Subword Tokenization.**

---
## 4 · Modern Subword Tokenization

Why didn't word-level tokenization survive the LLM era?
1. **Vocabulary Bloom:** Can't store 1M+ words in GPU memory.
2. **OOV (Out-of-Vocabulary):** If the model sees 'Unbelievable' but only trained on 'Believable', it's stuck.

**Solution:** Break words into meaningful sub-units.

### 4.1 · Byte-Pair Encoding (BPE)
*Used by: GPT-2, GPT-3, GPT-4, Llama 2/3.*

**Logic:**
1. Split words into individual bytes/characters.
2. Count the most frequent adjacent pair (e.g., 't' + 'h').
3. Merge them into a single token ('th').
4. Repeat until you reach your target `vocab_size`.

In [ ]:
from collections import Counter
import re
from typing import Dict, Tuple

def get_stats(vocab: Dict[str, int]) -> Counter:
    """Counts frequency of adjacent symbol pairs."""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols)-1):
            pairs[symbols[i], symbols[i+1]] += freq
    return pairs

def merge_vocab(pair: Tuple[str, str], v_in: Dict[str, int]) -> Dict[str, int]:
    """Merges a symbol pair into a single symbol in the vocabulary."""
    v_out = {}
    bigram = re.escape(' '.join(pair))
    p = re.compile(r'(?<!\S)' + bigram + r'(?!\S)')
    for word in v_in:
        w_out = p.sub(''.join(pair), word)
        v_out[w_out] = v_in[word]
    return v_out

# Initial Vocab (words split into chars space-separated)
vocab = {'l o w </w>' : 5, 'l o w e r </w>' : 2, 'n e w e s t </w>': 6, 'w i d e s t </w>': 3}
num_merges = 5

for i in range(num_merges):
    pairs = get_stats(vocab)
    if not pairs: break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"Merge {i+1}: {best} -> freq={pairs[best]}")
    print(f"Current state: {vocab}\n")

### 4.2 · WordPiece vs. Unigram (SentencePiece)

#### WordPiece (BERT, DistilBERT)
Similar to BPE but with a twist: instead of most frequent pair, it picks the pair that **maximizes likelihood**. 
$$Score = \frac{P(AB)}{P(A)P(B)}$$

#### Unigram / SentencePiece (T5, ALBERT, Llama 3)
It works in reverse! It starts with a HUGE vocabulary and **removes** pieces that don't contribute much to the probability of the corpus.

### 🎓 Masterclass Trick: Byte-Fallback

Older tokenizers (like BERT's) used an `[UNK]` token for any character not in their vocabulary (e.g., rare emojis like 🫠 or non-Latin scripts). 

**SentencePiece** (used by Llama models) solved this with **Byte-Fallback**:
1. The vocabulary pre-includes the **256 individual UTF-8 bytes**.
2. If a character is Out-of-Vocabulary, the tokenizer falls back to encoding it as its raw byte sequence.
3. **Impact:** No more `[UNK]` tokens. Every possible character string can be encoded perfectly, which is critical for multilingual and code-heavy datasets.

### 4.3 · Subword Regularization for Robustness

### 🎓 Socratic Deep Check
> *"We have spent this entire chapter treating tokenization as a deterministic function: one input, one output. But what if the 'correct' tokenization doesn't exist? The word 'unbelievable' could be tokenized as `['un', 'believ', 'able']`, or `['un', 'believe', 'able']`, or `['unbeliev', 'able']`. Each is a valid decomposition. If we always pick the same one, are we leaving information on the table?"*

---

#### The Core Idea: Tokenization as Data Augmentation

Standard BPE and WordPiece are **deterministic**: given a word, they always produce the exact same subword sequence. This means the model only ever sees one possible segmentation of each word, and it can become **brittle** — overfitting to that specific decomposition.

**Subword Regularization** (Kudo, 2018) reframes tokenization as a **stochastic process**. Instead of committing to a single 'best' segmentation, we **sample from multiple valid segmentations** during training. This acts as a powerful form of **data augmentation at the tokenization layer itself**, forcing the model to learn meaning from *multiple representations* of the same word.

**Why does this work?**
- The model can no longer memorize a fixed token-to-meaning mapping.
- It must learn **compositional understanding** — inferring meaning from varying subword contexts.
- This is analogous to how **dropout** prevents co-adaptation between neurons: subword regularization prevents co-adaptation between token positions.

There are two major implementations of this idea:

---

#### 4.3.1 · BPE Dropout (Provilkov et al., 2020)

Standard BPE applies merge operations greedily in a fixed order. **BPE Dropout** introduces a simple but devastating perturbation: at each merge step during tokenization, **randomly skip (drop) the merge with probability `p`**.

| Step | Standard BPE | BPE Dropout (`p=0.1`) |
|------|-------------|----------------------|
| Input | `u n b e l i e v a b l e` | `u n b e l i e v a b l e` |
| Merge 1 | `u n b e l ie v a b l e` | `u n b e l ie v a b l e` |
| Merge 2 | `u n b e liev a b le` | ⚡ *Skip! (dropped)* |
| Merge 3 | `un b eliev ab le` | `u n b e liev a b le` |
| ... | `un believ able` | `un b eliev ab le` |
| **Result** | `['un', 'believ', 'able']` | `['un', 'b', 'eliev', 'ab', 'le']` |

**Key Properties:**
- **`p = 0`** → Deterministic (standard BPE). This is used at **inference** time.
- **`p = 0.1`** → Recommended default for training. Provides a good balance between diversity and coherence.
- **`p = 1.0`** → Every merge is skipped, producing a pure character-level tokenization.
- The result is that the **same sentence produces different token sequences on every training epoch**, providing free data augmentation with zero computational overhead.

---

#### 4.3.2 · Unigram Sampling (Kudo, 2018)

The Unigram model (used by SentencePiece in T5, ALBERT, and Llama 3) is *inherently probabilistic*. It maintains a probability for every subword piece in the vocabulary, and the probability of a segmentation $\mathbf{x} = (x_1, x_2, \ldots, x_k)$ is:

$$P(\mathbf{x}) = \prod_{i=1}^{k} P(x_i)$$

The **Viterbi** (most probable) segmentation is what you get during deterministic inference. But during training, we can **sample** from the $n$-best segmentations according to their probabilities.

**Example:** For the word `'unbelievable'`, the Unigram model might have:

| Segmentation | Log-Probability |
|---|---|
| `['▁un', 'believ', 'able']` | -8.23 (Most Probable → Viterbi) |
| `['▁un', 'believe', 'able']` | -8.91 |
| `['▁unbeliev', 'able']` | -9.14 |
| `['▁un', 'bel', 'iev', 'able']` | -10.67 |

With **`nbest_size > 1`** and a **temperature** parameter `alpha`:
- **`alpha = 0`** → Uniform sampling from the top-k (maximum diversity).
- **`alpha = 1.0`** → Sample proportional to the true probabilities (calibrated).
- **`alpha → ∞`** → Always pick the Viterbi path (deterministic).

> **🎓 The Masterclass Insight:** BPE Dropout and Unigram Sampling achieve the same goal — **injecting noise at the tokenization layer** — but from opposite directions. BPE Dropout *removes* structure (skipping merges). Unigram Sampling *adds* stochasticity (sampling from a probability distribution). Both are forms of **regularization**, and both produce measurable improvements on low-resource translation and noisy text classification tasks.

---

#### 4.3.3 · Practical Implementation

In [ ]:
# =============================================================================
# 4.3.3 · Subword Regularization in Practice
# =============================================================================
# We demonstrate both approaches using industry-standard libraries.

# --- Method 1: BPE Dropout with HuggingFace `tokenizers` ----------------------
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import json
import tempfile
import os

# Train a small BPE tokenizer for demonstration
training_corpus = [
    "The unbelievable power of subword tokenization is transforming NLP.",
    "Regularization prevents overfitting and improves generalization.",
    "Dropout is a simple but powerful technique for neural network training.",
    "Subword algorithms break words into meaningful units for better coverage.",
    "The unbelievable advancements in natural language processing continue.",
    "Natural language processing relies on tokenization and subword models.",
    "Better tokenization leads to better language understanding and generation.",
]

bpe_tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
bpe_tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]"], vocab_size=200)
bpe_tokenizer.train_from_iterator(training_corpus, trainer)

# Save → inject dropout into config → reload
# This is the production pattern for enabling BPE Dropout.
with tempfile.TemporaryDirectory() as tmpdir:
    path = os.path.join(tmpdir, 'tokenizer.json')
    bpe_tokenizer.save(path)
    
    # Inject dropout=0.1 into the saved BPE model config
    with open(path, 'r') as f:
        config = json.load(f)
    config['model']['dropout'] = 0.1  # ← The key parameter
    with open(path, 'w') as f:
        json.dump(config, f)
    
    # Reload with dropout active
    bpe_dropout_tokenizer = Tokenizer.from_file(path)

test_word = "unbelievable"
print("=" * 60)
print("BPE DROPOUT DEMONSTRATION")
print("=" * 60)
print(f"Input: '{test_word}'\n")
print("Multiple tokenizations of the SAME word (dropout=0.1):")
for i in range(5):
    tokens = bpe_dropout_tokenizer.encode(test_word).tokens
    print(f"  Run {i+1}: {tokens}")

print("\n→ Notice: each run may produce a DIFFERENT segmentation.")
print("  At inference time, set dropout=0.0 for deterministic output.")

# --- Method 2: Unigram Sampling with SentencePiece ---------------------------
# SentencePiece natively supports nbest_size and alpha for Unigram sampling.
# Below is the API pattern you would use with a trained .model file.

print("\n" + "=" * 60)
print("UNIGRAM SAMPLING (SentencePiece API Pattern)")
print("=" * 60)

sentencepiece_example = '''
import sentencepiece as spm

# Load a pre-trained SentencePiece model (e.g., from Llama, T5)
sp = spm.SentencePieceProcessor(model_file='my_model.model')

# --- Deterministic (Inference) ---
# Viterbi decoding: always returns the single most probable segmentation
tokens = sp.encode('unbelievable', out_type=str)
# → ['▁un', 'believ', 'able']

# --- Stochastic (Training with Subword Regularization) ---
# nbest_size: sample from top-k segmentations (use -1 for all)
# alpha:       temperature (0.0 = uniform, 1.0 = proportional to prob)
tokens = sp.encode(
    'unbelievable',
    out_type=str,
    enable_sampling=True,   # ← Enable subword regularization
    nbest_size=-1,          # ← Sample from ALL valid segmentations
    alpha=0.1               # ← Smoothing temperature
)
# → Different result each call: ['▁un', 'believe', 'able']
#                            or: ['▁unbeliev', 'able']
#                            or: ['▁un', 'bel', 'iev', 'able']
'''
print(sentencepiece_example)
print("💡 Key Takeaway:")
print("   During TRAINING  → enable_sampling=True  (regularization ON)")
print("   During INFERENCE → enable_sampling=False (deterministic, reproducible)")

---
## 5 · Production Hands-on: Building a Tokenizer

In industry, we use the `tokenizers` library (written in Rust) because performance is critical. Training a BPE tokenizer on a 1GB dataset shouldn't take hours.

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Initialize an empty BPE model
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 2. Trainer Config
trainer = BpeTrainer(special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"], 
                    vocab_size=1000) 

# 3. Training Data
data = [
    "The quick brown fox jumps over the lazy dog.",
    "Tokenization is the foundation of NLP engineering.",
    "Modern subword algorithms handle out-of-vocabulary words intelligently."
]

# 4. Train
tokenizer.train_from_iterator(data, trainer)

# 5. Test it!
test_sentence = "The foundation of NLP is tokenization."
output = tokenizer.encode(test_sentence)
print(f"Sentence: {test_sentence}")
print(f"Tokens:   {output.tokens}")
print(f"IDs:      {output.ids}")

---
## ✅ Knowledge Check

### Q1: The OOV Paradox
Which algorithm handles the word "Hyper-loop" better if it was never in the training set: **Word-level** or **Subword (BPE)**?

<details><summary>👉 Answer</summary>
Subword (BPE). Word-level would flag it as [UNK] (Unknown). BPE would likely break it into ["Hyper", "-", "loop"], allowing the model to aggregate the meanings of those pieces.
</details>

### Q2: Mathematical Efficiency
Why does TF-IDF use the Logarithm for the Inverse Document Frequency? 

<details><summary>👉 Answer</summary>
Without the log, IDF values would grow linearly with the number of documents. For a corpus of 1 million docs where a word appears in only 1, the multiplier would be 1,000,000. This would drastically over-emphasize rare words. The log dampens this growth, making the weights manageable.
</details>

### Q3: Byte-Fallback (Senior Level)
How does Byte-Fallback help in preventing `[UNK]` for emojis like 🫠?

<details><summary>👉 Answer</summary>
If '🫠' is not in the vocab, the tokenizer converts it to its UTF-8 hex bytes (e.g., `\xf0\x9f\xaa\xa0`). Since all 256 bytes (0-255) are in the SentencePiece vocabulary, it can always represent the emoji as a sequence of 4 byte-tokens.
</details>

---

## 🔨 Practical Challenge

**Challenge:** Use the `augmentor` class you built to generate 3 augmented versions of the sentence: *"Artificial Intelligence is transforming the global economy at an unprecedented pace."*

--- 
**Next →** [NLP 02: Word Embeddings & Word2Vec](02_word_embeddings_and_word2vec.ipynb)